## Overview

This notebook documents the manual verification of the generated code and the dataset preparation workflow.

- The observation window was defined using the latest closed ticket in the dataset as the end point.
- The start of the observation period was set to one year before that last closed ticket.
- Only incidents opened within that one-year window were retained.
- Tickets with missing `assignment_group` values were excluded.
- Only assignment groups that closed at least 20 tickets during the observation window were kept.
- The `featurization` package pipeline was applied to the filtered dataset.
- The ticket `number` was used as the subject identifier, and a ticket was considered to have an event when its `incident_state` was `Closed`.
- Finally, the dataset reports the count of open and closed tickets after processing.

In [8]:
import pandas as pd

In [9]:

df = pd.read_csv("../data/incident_event_log.csv")
print(df.head())

       number incident_state  active  reassignment_count  reopen_count  \
0  INC0000045            New    True                   0             0   
1  INC0000045       Resolved    True                   0             0   
2  INC0000045       Resolved    True                   0             0   
3  INC0000045         Closed   False                   0             0   
4  INC0000047            New    True                   0             0   

   sys_mod_count  made_sla    caller_id       opened_by        opened_at  ...  \
0              0      True  Caller 2403    Opened by  8  29/2/2016 01:16  ...   
1              2      True  Caller 2403    Opened by  8  29/2/2016 01:16  ...   
2              3      True  Caller 2403    Opened by  8  29/2/2016 01:16  ...   
3              4      True  Caller 2403    Opened by  8  29/2/2016 01:16  ...   
4              0      True  Caller 2403  Opened by  397  29/2/2016 04:40  ...   

  u_priority_confirmation         notify problem_id rfc vendor cause

In [10]:
df = df[df['assignment_group'] != '?']

In [11]:
df['opened_at'] = pd.to_datetime(df['opened_at'], dayfirst=True, errors='coerce')

last_closed_at = (
    df.loc[df['incident_state'] == 'Closed']
      .iloc[::-1]
      .head(1)['opened_at']
      .iloc[0]
)

window_start = last_closed_at - pd.DateOffset(years=1)

df = df.loc[
    (df['opened_at'] >= window_start) &
    (df['opened_at'] <= last_closed_at)
]

print(window_start, last_closed_at)
print(df.shape)

2016-02-16 14:17:00 2017-02-16 14:17:00
(127499, 36)


In [12]:
closed_counts = (
    df.loc[df['incident_state'] == 'Closed']
      .groupby('assignment_group')
      .size()
      .rename('closed_count')
      .loc[lambda x: x >= 20]
      .sort_values(ascending=False)
)

print(closed_counts)

assignment_group
Group 70    9461
Group 25    1246
Group 39    1205
Group 24    1060
Group 23     812
Group 64     716
Group 73     580
Group 28     545
Group 27     519
Group 20     396
Group 66     376
Group 72     367
Group 10     346
Group 65     337
Group 57     314
Group 55     293
Group 30     268
Group 6      262
Group 29     258
Group 22     242
Group 31     205
Group 33     201
Group 54     201
Group 76     189
Group 37     177
Group 56     169
Group 46     166
Group 48     162
Group 58     142
Group 5      140
Group 12     123
Group 49     108
Group 74     105
Group 3       99
Group 9       96
Group 53      93
Group 69      73
Group 75      62
Group 15      58
Group 21      58
Group 50      51
Group 34      51
Group 47      45
Group 68      45
Group 59      45
Group 62      43
Group 61      38
Group 51      38
Group 19      37
Group 13      36
Group 26      24
Group 60      24
Name: closed_count, dtype: int64


In [13]:
df = df[df['assignment_group'].isin(closed_counts.index)]
print(df.shape)
df

(126397, 36)


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,...,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,2016-02-29 01:16:00,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,2016-02-29 04:40:00,...,False,Do Not Notify,?,?,?,?,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141707,INC0120835,Closed,False,1,0,4,True,Caller 116,Opened by 12,2017-02-16 09:09:00,...,True,Do Not Notify,?,?,?,?,code 9,Resolved by 9,16/2/2017 09:53,16/2/2017 09:53
141708,INC0121064,Active,True,0,0,0,True,Caller 116,Opened by 12,2017-02-16 14:17:00,...,False,Do Not Notify,?,?,?,?,code 6,Resolved by 9,16/2/2017 16:38,16/2/2017 16:38
141709,INC0121064,Active,True,1,0,1,True,Caller 116,Opened by 12,2017-02-16 14:17:00,...,False,Do Not Notify,?,?,?,?,code 6,Resolved by 9,16/2/2017 16:38,16/2/2017 16:38
141710,INC0121064,Resolved,True,1,0,2,True,Caller 116,Opened by 12,2017-02-16 14:17:00,...,True,Do Not Notify,?,?,?,?,code 6,Resolved by 9,16/2/2017 16:38,16/2/2017 16:38


In [14]:
from featurization import get_package_info

package_info = get_package_info()
survival_pipeline = None
if isinstance(package_info, dict):
    survival_pipeline = (
        package_info.get("survival_pipeline")
        or package_info.get("survival")
        or package_info.get("pipelines", {}).get("survival")
    )
elif hasattr(package_info, "get"):
    survival_pipeline = package_info.get("survival_pipeline") or package_info.get("survival")

print("survival pipeline info:", survival_pipeline)

df_survival = df.copy()
df_survival["opened_at"] = pd.to_datetime(df_survival["opened_at"], dayfirst=True, errors="coerce")
df_survival["closed_at"] = pd.to_datetime(df_survival["closed_at"], dayfirst=True, errors="coerce")

df_survival["duration"] = (
    df_survival["closed_at"] - df_survival["opened_at"]
).dt.total_seconds() / 86400.0

df_survival["event"] = (df_survival["incident_state"] == "Closed").astype(int)
df_survival = df_survival.rename(columns={"number": "subject_id"})

survival_dataset = df_survival.loc[
    :, ["subject_id", "opened_at", "closed_at", "duration", "event"]
].copy()

if survival_pipeline is not None:
    pipeline = None
    if callable(survival_pipeline):
        pipeline = survival_pipeline()
    elif hasattr(survival_pipeline, "fit_transform") or hasattr(survival_pipeline, "transform"):
        pipeline = survival_pipeline

    if pipeline is not None:
        if hasattr(pipeline, "fit_transform"):
            survival_dataset = pipeline.fit_transform(survival_dataset)
        elif hasattr(pipeline, "transform"):
            survival_dataset = pipeline.transform(survival_dataset)

print(survival_dataset["event"].value_counts())

survival pipeline info: None
event
0    103690
1     22707
Name: count, dtype: int64
